# Eastport Pipeline — Maine Nearshore

End-to-end audit for wind/solar/tidal with fishing constraints.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## 1. Config audit

- [ ] Load `data/eastport/config.yaml`
- [ ] Validate with `validate_config()`
- [ ] Print enabled technologies and economic assumptions

In [ ]:
from fleximorpv2.config import load_config, validate_config

config = load_config("eastport")
validate_config(config)
print(f"Site: {config.name}")
print(f"Coords: {config.coordinates}")
print(f"Techs: {config.get_enabled_technologies()}")
print(f"Discount rate: {config.economic.get('discount_rate')}")

## 2. Resource data audit

- [ ] Inspect synthetic weather means (wind, irradiance, wave if applicable)
- [ ] Confirm units: m/s, W/m², m, s
- [ ] Note whether live API or synthetic path is used

In [ ]:
from fleximorpv2.utils.data_loader import APIDataLoader

loader = APIDataLoader(config)
resource = loader.load_weather_data(
    coordinates=config.coordinates,
    technologies=config.get_enabled_technologies(),
)
print("Wind mean (m/s):", float(resource.wind_speed.mean()))
print("Solar mean (W/m²):", float(resource.solar_irradiance.mean()))
if hasattr(resource, "wave_height") and resource.wave_height is not None:
    print("Wave Hs mean (m):", float(resource.wave_height.mean()))

## 3. Baseline optimization

- [ ] Run with fixed seed / reproducible method
- [ ] Record LCOE, NPV, capacities, annual energy
- [ ] **Verify production target units (kWh vs MWh)**

In [ ]:
from fleximorpv2.baseline_optimization import BaselineOptimization

config.uncertainty["monte_carlo_runs"] = 10
baseline = BaselineOptimization(config)
results = baseline.optimize("production", 200_000, method="scipy")

assert_positive(results.financial_metrics["lcoe"], label="LCOE")
assert_energy_balance(
    results.technical_metrics["annual_energy"],
    results.technical_metrics["total_capacity"],
    results.technical_metrics["capacity_factor"],
)
print(results.financial_metrics)
print(results.technical_metrics)
print(results.technology_capacities)
optimal_design = results.optimal_design

## 4. Uncertainty analysis

- [ ] Monte Carlo n=50 then n=500 locally
- [ ] Latin Hypercube comparison
- [ ] Mean LCOE same order of magnitude as baseline

In [ ]:
from fleximorpv2.uncertainty_analysis import UncertaintyAnalysis

analyzer = UncertaintyAnalysis(config)
uncertainty = analyzer.analyze_uncertainty(
    baseline_design=optimal_design,
    sampling_method="monte_carlo",
    reoptimize=False,
)
print(uncertainty.mean_performance)
print(uncertainty.risk_metrics)

# TODO: run latin_hypercube and compare_sampling_methods

## 5. Extended pipeline (optional)

- [ ] Flexible design (`fleximorpv2/flexible_design.py`)
- [ ] Sensitivity analysis
- [ ] Save plots to `notebooks/sites/outputs/`

In [ ]:
# TODO: flexible design + sensitivity
pass